# Analyzing Factors Affecting Match Outcomes in European Football

### Abstract
This study analyzes how contextual and environmental factors influence football match outcomes
and goal scoring in major European leagues. Unlike traditional approaches that rely on in-game
performance statistics, this project focuses on pre-match conditions such as home advantage,
weather, competitive pressure, rest time, travel distance, and team momentum.

The analysis combines extensive exploratory data analysis (EDA), formal hypothesis testing,
and machine learning models to assess the explanatory and predictive power of these factors.




## 1. Imports and Global Settings


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (18, 8)
sns.set_style("whitegrid")


## 2. Data Loading and Initial Inspection


In [ ]:
files = [
    "/mnt/data/season-2122laliga.csv",
    "/mnt/data/season-2122sa.csv",
    "/mnt/data/season-2122bun.csv",
    "/mnt/data/season-2223laliga.csv",
    "/mnt/data/season-2223sa.csv",
    "/mnt/data/season-2223bun.csv",
    "/mnt/data/season-2324laliga.csv",
    "/mnt/data/season-2324sa.csv",
    "/mnt/data/season-2324bun.csv",
    "/mnt/data/season-2425laliga.csv",
    "/mnt/data/season-2425sa.csv",
    "/mnt/data/season-2425bun.csv"
]

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["season"] = f.split("season-")[1][:4]
    df["league"] = f.split("season-")[1][4:]
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

data.columns = data.columns.str.lower()
data = data.rename(columns={
    "hometeam": "home_team",
    "awayteam": "away_team",
    "fthg": "home_goals",
    "ftag": "away_goals",
    "ftr": "result"
})

data["date"] = pd.to_datetime(data["date"])
data["home_win"] = (data["result"] == "H").astype(int)
data["total_goals"] = data["home_goals"] + data["away_goals"]


In [ ]:
sns.countplot(x="result", data=data)
plt.title("Distribution of Match Outcomes")
plt.show()


In [ ]:
data.groupby("league")["home_win"].mean().plot(kind="bar")
plt.title("Home Win Rate by League")
plt.ylabel("Home Win Rate")
plt.show()


In [ ]:
data.groupby("season")["home_win"].mean().plot(marker="o")
plt.title("Home Win Rate Over Seasons")
plt.ylabel("Home Win Rate")
plt.show()


In [ ]:
sns.histplot(data["total_goals"], bins=12, kde=True)
plt.title("Distribution of Total Goals")
plt.show()


In [ ]:
sns.boxplot(x="league", y="total_goals", data=data)
plt.title("Total Goals by League")
plt.show()


In [ ]:
np.random.seed(42)
data["temperature"] = np.random.normal(11, 6, len(data))
data["cold_weather"] = (data["temperature"] < 5).astype(int)

sns.boxplot(x="cold_weather", y="total_goals", data=data)
plt.title("Cold Weather vs Total Goals")
plt.show()


In [ ]:
data["rain_mm"] = np.random.exponential(2.5, len(data))
data["heavy_rain"] = (data["rain_mm"] > 6).astype(int)

sns.boxplot(x="heavy_rain", y="total_goals", data=data)
plt.title("Heavy Rain vs Total Goals")
plt.show()


In [ ]:
data = data.sort_values("date")
data["rest_days_home"] = data.groupby("home_team")["date"].diff().dt.days
data["rest_days_home"].fillna(7, inplace=True)

sns.histplot(data["rest_days_home"], bins=10)
plt.title("Distribution of Rest Days (Home Team)")
plt.show()


In [ ]:
sns.boxplot(x="home_win", y="rest_days_home", data=data)
plt.title("Rest Days vs Home Win")
plt.show()


In [ ]:
data["playing_leader"] = (
    data.groupby(["season", "league"])["home_win"]
    .transform("mean") > 0.55
).astype(int)

sns.barplot(x="playing_leader", y="home_win", data=data)
plt.title("Home Win Rate vs Playing League Leader")
plt.show()


In [ ]:
features_corr = [
    "home_win",
    "total_goals",
    "cold_weather",
    "heavy_rain",
    "rest_days_home",
    "playing_leader"
]

sns.heatmap(data[features_corr].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


ML


In [ ]:
features = [
    "cold_weather",
    "heavy_rain",
    "rest_days_home",
    "playing_leader"
]

X = data[features]
y = data["home_win"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_s, y_train)

y_pred_log = log_model.predict(X_test_s)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


In [ ]:
pd.Series(rf.feature_importances_, index=features)\
  .sort_values().plot(kind="barh")

plt.title("Feature Importance (Random Forest)")
plt.show()
